# Credit Risk Model & Credit Strategy (Amex Default Prediction)

**How to use this notebook**
1. Runtime → Change runtime type → CPU is fine (no GPU needed).
2. Run the cells **in order**, top to bottom.
3. Every important result is saved to Google Drive in `MyDrive/Credit_Risk_Project/`.
   If Colab disconnects, re-run **Section 0** (setup) and then continue from the section you were on.
   Each section reloads what it needs from Drive.
4. The grid searches save their table after every model, so if they stop, just re-run the cell and they continue where they left off.

All slide material (tables as CSV/Excel, charts as PNG) ends up in `MyDrive/Credit_Risk_Project/slides/`.

## 0. Setup (run this every time you open the notebook)

In [ ]:
!pip -q install shap openpyxl

import os, sys, glob, json, pickle, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR  = "/content/drive/MyDrive/Credit_Risk_Project"   # results (persist)
    DATA_DIR = "/content/data"                               # raw Kaggle files (big, temporary)
else:
    OUT_DIR  = os.environ.get("OUT_DIR", "./Credit_Risk_Project")
    DATA_DIR = os.environ.get("DATA_DIR", "./data")

SLIDES = os.path.join(OUT_DIR, "slides")
for d in [OUT_DIR, SLIDES, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

TRAIN_DATA   = os.path.join(DATA_DIR, "train_data.csv")
TRAIN_LABELS = os.path.join(DATA_DIR, "train_labels.csv")
SEED = 42

# Amex categorical features (from the Kaggle data description)
CAT_ALL = ["B_30", "B_38", "D_114", "D_116", "D_117", "D_120", "D_126", "D_63", "D_64", "D_66", "D_68"]
CATEGORY_NAMES = {"D": "Delinquency", "S": "Spend", "P": "Payment", "B": "Balance", "R": "Risk"}

def category_of(col):
    """Map a raw or engineered feature name to its Amex category."""
    return CATEGORY_NAMES.get(col.split("_")[0], "Other")

def p(name):
    return os.path.join(OUT_DIR, name)

def s(name):
    return os.path.join(SLIDES, name)

print("Results folder:", OUT_DIR)

from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
GRID_XGB = p("grid_xgb.csv")
GRID_NN = p("grid_nn.csv")

def load_splits():
    tr, t1, t2 = [pd.read_parquet(p(f"{n}.parquet")) for n in ["train", "test1", "test2"]]
    return tr, t1, t2
def summarize_grid(res):
    r = res.copy()
    a = r[["AUC Train", "AUC Test 1", "AUC Test 2"]]
    r["Avg AUC"] = a.mean(axis=1)
    r["Std AUC"] = a.std(axis=1)
    r["Gap (Train-Test2)"] = r["AUC Train"] - r["AUC Test 2"]
    return r

def pick_best(r, max_gap=0.02):
    ok = r[r["Gap (Train-Test2)"] <= max_gap]
    return ok["Avg AUC"].idxmax() if len(ok) else r["Gap (Train-Test2)"].idxmin()

def grid_scatter(r, best, title, fname):
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ax[0].scatter(r["Avg AUC"], r["Std AUC"], c="#4C6EF5", alpha=0.7)
    ax[0].scatter(r.loc[best, "Avg AUC"], r.loc[best, "Std AUC"], c="#E03131", s=140, label="Chosen model")
    ax[0].set_xlabel("Average AUC (Train, Test 1, Test 2)"); ax[0].set_ylabel("Std. Dev. of AUC")
    ax[0].set_title("Average vs. Std. Dev. of AUC")
    ax[1].scatter(r["AUC Train"], r["AUC Test 2"], c="#4C6EF5", alpha=0.7)
    ax[1].scatter(r.loc[best, "AUC Train"], r.loc[best, "AUC Test 2"], c="#E03131", s=140, label="Chosen model")
    lo, hi = min(r["AUC Train"].min(), r["AUC Test 2"].min()), max(r["AUC Train"].max(), r["AUC Test 2"].max())
    ax[1].plot([lo, hi], [lo, hi], "--", c="gray", lw=1, label="Train = Test 2")
    ax[1].set_xlabel("AUC Train"); ax[1].set_ylabel("AUC Test 2"); ax[1].set_title("Train vs. Test 2 AUC")
    for a_ in ax:
        a_.legend(); a_.spines[["top", "right"]].set_visible(False)
    fig.suptitle(title, fontsize=14); plt.tight_layout(); plt.savefig(s(fname), dpi=200); plt.show()

def rank_order(pd_tr, pd_t1, pd_t2, y_tr, y_t1, y_t2, title, fname, n_bins=10):
    # bins defined on TRAIN, same cut-offs applied to the tests
    _, edges = pd.qcut(pd_tr, q=n_bins, retbins=True, duplicates="drop")
    edges[0], edges[-1] = -np.inf, np.inf
    labels_ = [f"{i+1}" for i in range(len(edges) - 1)]
    out = []
    for name, pr, y in [("Train", pd_tr, y_tr), ("Test 1", pd_t1, y_t1), ("Test 2", pd_t2, y_t2)]:
        b = pd.cut(pr, edges, labels=labels_)
        t = pd.DataFrame({"bin": b, "y": np.asarray(y)}).groupby("bin", observed=False)["y"].agg(["size", "mean"])
        t.columns = [f"{name} # Obs", f"{name} Default Rate"]; out.append(t)
    tab = pd.concat(out, axis=1)
    ranges = [f"{max(edges[i],0):.3f}-{min(edges[i+1],1):.3f}" for i in range(len(edges)-1)]
    tab.insert(0, "PD Range (Train cut-offs)", ranges)
    tab.index.name = "Score Bin (1 = lowest PD)"
    x = np.arange(len(tab)); wdt = 0.27
    fig, ax = plt.subplots(figsize=(11, 5))
    for k, (name, col) in enumerate([("Train", "#4C6EF5"), ("Test 1", "#74C0FC"), ("Test 2", "#2B8A3E")]):
        ax.bar(x + (k-1)*wdt, tab[f"{name} Default Rate"], wdt, label=name, color=col)
    ax.set_xticks(x); ax.set_xticklabels(tab.index)
    ax.set_xlabel("Score bin (deciles of predicted PD, 1 = safest)"); ax.set_ylabel("Actual default rate")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_title(title, fontsize=14); ax.legend(); ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.savefig(s(fname), dpi=200); plt.show()
    return tab

## 1. Download the data from Kaggle (only once)

Before running this cell:
1. Go to https://www.kaggle.com/competitions/amex-default-prediction and click **Join Competition / I Understand and Accept** (required, otherwise download fails with 403).
2. Kaggle → your profile picture → **Settings** → **API** → **Create New Token** and copy the token it shows.
3. Run the cell and paste the token in the box when asked (it stays hidden).

`train_data.csv` is about 16 GB, so the download + unzip takes ~10 to 20 minutes. It goes to Colab's local disk (not Drive).

In [ ]:
if not (os.path.exists(TRAIN_DATA) and os.path.exists(TRAIN_LABELS)):
    if IN_COLAB:
        from getpass import getpass
        # Paste the API token Kaggle showed you (Settings -> API -> Create New Token). It stays hidden.
        os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token and press Enter: ").strip()
        !pip -q install -U kaggle
        !kaggle competitions download -c amex-default-prediction -f train_labels.csv -p {DATA_DIR}
        !kaggle competitions download -c amex-default-prediction -f train_data.csv -p {DATA_DIR}
        for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
            !unzip -o -q "{z}" -d {DATA_DIR}
            os.remove(z)
    else:
        raise FileNotFoundError("Put train_data.csv and train_labels.csv in " + DATA_DIR)
print(os.listdir(DATA_DIR))

## 2–4. Read the data and build the development sample

* Pick **10% of the feature columns at random** from `train_data.csv`. `customer_ID` (key) and `S_2` (statement date, needed to know which month each row is) are always kept, they are not counted as features.
* Pick **20% of the customers at random** from `train_labels.csv`.
* Read `train_data.csv` **in chunks**, keep only the sampled columns, and keep only rows of the sampled customers. This is the same as merging the 20% labels sample with the 10% train_data, but it never loads the 16 GB file into memory.
* Merge with the labels and **save** to Drive.

Guard rule (our decision): the random draw is repeated until it contains at least one Spend (`S_`) feature and one Balance (`B_`) feature (needed for the revenue formula) and at least one categorical feature (needed for one-hot encoding).

In [ ]:
all_cols = pd.read_csv(TRAIN_DATA, nrows=0).columns.tolist()
feature_pool = [c for c in all_cols if c not in ["customer_ID", "S_2"]]
n_feat = int(round(0.10 * len(feature_pool)))

rng = np.random.RandomState(SEED)
draws = 0
while True:
    draws += 1
    chosen = sorted(rng.choice(feature_pool, n_feat, replace=False).tolist())
    has_S   = any(c.startswith("S_") for c in chosen)
    has_B   = any(c.startswith("B_") for c in chosen)
    has_cat = any(c in CAT_ALL for c in chosen)
    if has_S and has_B and has_cat:
        break
print(f"{len(feature_pool)} features in raw data -> randomly chose {n_feat} (10%) after {draws} draw(s):")
print(chosen)

labels = pd.read_csv(TRAIN_LABELS)
labels_sample = labels.sample(frac=0.20, random_state=SEED).reset_index(drop=True)
print(f"Labels: {len(labels):,} customers -> 20% sample = {len(labels_sample):,}")

ids = set(labels_sample["customer_ID"])
usecols = ["customer_ID", "S_2"] + chosen
parts, total_rows = [], 0
t0 = time.time()
for i, chunk in enumerate(pd.read_csv(TRAIN_DATA, usecols=usecols, chunksize=1_000_000)):
    total_rows += len(chunk)
    parts.append(chunk[chunk["customer_ID"].isin(ids)])
    print(f"chunk {i+1}: read {total_rows:,} rows so far ({time.time()-t0:.0f}s)")
data_10pct = pd.concat(parts, ignore_index=True)[usecols]
del parts

dev = data_10pct.merge(labels_sample, on="customer_ID", how="inner", validate="many_to_one")

# Checks: every sampled customer must be found, one target per customer
print("\nRows in train_data:", f"{total_rows:,}")
print("Rows in development sample:", f"{len(dev):,}")
print("Customers in development sample:", f"{dev['customer_ID'].nunique():,}", "(should equal", f"{len(labels_sample):,})")
assert dev["customer_ID"].nunique() == len(labels_sample)
assert dev.groupby("customer_ID")["target"].nunique().max() == 1

dev.to_parquet(p("dev_sample.parquet"), index=False)
json.dump({"chosen_features": chosen}, open(p("chosen_features.json"), "w"))
print("Saved:", p("dev_sample.parquet"))

## Slides 2–3 tables: data summary (# months of history) and raw feature categories

In [ ]:
dev = pd.read_parquet(p("dev_sample.parquet"))
chosen = json.load(open(p("chosen_features.json")))["chosen_features"]

n_months = dev.groupby("customer_ID").agg(n_months=("S_2", "size"), target=("target", "first"))
t = (n_months.groupby("n_months")["target"].agg(Obs="size", Default_Rate="mean")
     .sort_index(ascending=False).reset_index())
t["All Applications"] = t["n_months"].astype(str) + t["n_months"].map(lambda m: " Month" if m == 1 else " Months")
total = pd.DataFrame({"All Applications": ["Total"], "Obs": [len(n_months)], "Default_Rate": [n_months["target"].mean()]})
slide2 = pd.concat([t[["All Applications", "Obs", "Default_Rate"]], total], ignore_index=True)
slide2.columns = ["All Applications", "# Obs", "Default Rate"]
slide2.to_csv(s("slide02_months_of_data.csv"), index=False)
show = slide2.copy(); show["# Obs"] = show["# Obs"].map("{:,}".format); show["Default Rate"] = show["Default Rate"].map("{:.2%}".format)
print(show.to_string(index=False))
print("\nStatement dates:", dev["S_2"].min(), "to", dev["S_2"].max())

raw_cats = pd.Series([category_of(c) for c in chosen]).value_counts().rename_axis("Category").reset_index(name="# of features")
raw_cats.loc[len(raw_cats)] = ["Total", raw_cats["# of features"].sum()]
raw_cats.to_csv(s("slide03_raw_feature_categories.csv"), index=False)
print("\nRaw features by category:\n", raw_cats.to_string(index=False))
print("\nCategorical features in sample:", [c for c in chosen if c in CAT_ALL])

## 5. One-hot encoding

Each categorical feature becomes one 0/1 column per level. Missing values get their own `_nan` column (our decision, because "no value" can itself be informative). Columns that are always 0 are dropped.

In [ ]:
dev = pd.read_parquet(p("dev_sample.parquet"))
chosen = json.load(open(p("chosen_features.json")))["chosen_features"]
cats = [c for c in chosen if c in CAT_ALL]
nums = [c for c in chosen if c not in CAT_ALL]

# Slide 5: levels of each categorical variable
rows = []
for c in cats:
    vc = dev[c].astype("object").where(dev[c].notna(), "Missing").astype(str).value_counts()
    for lvl, n in vc.items():
        rows.append([c, lvl, n, n / len(dev)])
levels = pd.DataFrame(rows, columns=["Feature", "Level", "# Rows", "% Rows"])
levels.to_csv(s("slide05_categorical_levels.csv"), index=False)
print(levels.to_string(index=False))

# ---- One-hot encoding ----
for c in cats:
    # make codes like 1.0 / 2.0 into clean text labels "1" / "2"
    dev[c] = dev[c].map(lambda v: v if pd.isna(v) else (str(int(v)) if isinstance(v, (int, float, np.floating)) and float(v).is_integer() else str(v)))
dev_ohe = pd.get_dummies(dev, columns=cats, prefix=cats, dummy_na=True, dtype=np.int8)
ohe_cols = [c for c in dev_ohe.columns if any(c.startswith(k + "_") for k in cats) and c not in nums]
const = [c for c in ohe_cols if dev_ohe[c].nunique() == 1]
dev_ohe = dev_ohe.drop(columns=const)
ohe_cols = [c for c in ohe_cols if c not in const]

ohe_table = pd.DataFrame({"Original Feature": [next(k for k in cats if c.startswith(k + "_")) for c in ohe_cols],
                          "New Binary Column": ohe_cols,
                          "% of Rows = 1": [dev_ohe[c].mean() for c in ohe_cols]})
ohe_table.to_csv(s("slide05_one_hot_result.csv"), index=False)
print("\n", ohe_table.to_string(index=False))
print(f"\n{len(cats)} categorical features -> {len(ohe_cols)} binary columns")

dev_ohe.to_parquet(p("dev_sample_ohe.parquet"), index=False)
json.dump({"cats": cats, "nums": nums, "ohe_cols": ohe_cols}, open(p("column_groups.json"), "w"))

## 6. Feature engineering (account level, as of April 2018)

For each customer the most recent statement is treated as **April 2018** (month 1), the one before as month 2, and so on back to month 13 (April 2017). Customers with fewer months use what they have.

**Numeric features:** Ave 3/6/9/12, Min 3/6/12, Max 3/6/12, Sum 3/6/12, Apr_2018 (latest value), Chg_6 = (Apr18 − Oct17)/Oct17, Chg_12 = (Apr18 − Apr17)/Apr17.

**One-hot (binary) features:** Response_Rate 3/6/12 (% of months = 1), Ever_Response 3/12, Apr_2018.

Plus `N_Months` = number of statements the customer has.

In [ ]:
dev = pd.read_parquet(p("dev_sample_ohe.parquet"))
g = json.load(open(p("column_groups.json")))
nums, ohe_cols = g["nums"], g["ohe_cols"]

dev["S_2"] = pd.to_datetime(dev["S_2"])
dev = dev.sort_values(["customer_ID", "S_2"]).reset_index(drop=True)
dev["m"] = dev.groupby("customer_ID").cumcount(ascending=False) + 1   # 1 = latest = Apr 2018

feats = [dev.groupby("customer_ID").agg(target=("target", "first"), N_Months=("S_2", "size"))]

def window(w):
    return dev[dev["m"] <= w].groupby("customer_ID")

# ---- numeric ----
for w in [3, 6, 9, 12]:
    feats.append(window(w)[nums].mean().add_suffix(f"_Ave_{w}"))
for w in [3, 6, 12]:
    gw = window(w)[nums]
    feats.append(gw.min().add_suffix(f"_Min_{w}"))
    feats.append(gw.max().add_suffix(f"_Max_{w}"))
    feats.append(gw.sum(min_count=1).add_suffix(f"_Sum_{w}"))
latest = dev[dev["m"] == 1].set_index("customer_ID")
m7     = dev[dev["m"] == 7].set_index("customer_ID")
m13    = dev[dev["m"] == 13].set_index("customer_ID")
feats.append(latest[nums].add_suffix("_Apr_2018"))
chg6  = (latest[nums] - m7[nums].reindex(latest.index)) / m7[nums].reindex(latest.index)
chg12 = (latest[nums] - m13[nums].reindex(latest.index)) / m13[nums].reindex(latest.index)
feats.append(chg6.add_suffix("_Chg_6"))
feats.append(chg12.add_suffix("_Chg_12"))

# ---- one-hot / binary ----
if ohe_cols:
    for w in [3, 6, 12]:
        feats.append(window(w)[ohe_cols].mean().add_suffix(f"_Response_Rate_{w}"))
    for w in [3, 12]:
        feats.append(window(w)[ohe_cols].max().add_suffix(f"_Ever_Response_{w}"))
    feats.append(latest[ohe_cols].add_suffix("_Apr_2018"))

acct = pd.concat(feats, axis=1)
acct = acct.replace([np.inf, -np.inf], np.nan)            # divide by zero -> NaN
acct.index.name = "customer_ID"
acct = acct.reset_index()
feature_cols = [c for c in acct.columns if c not in ["customer_ID", "target"]]
acct[feature_cols] = acct[feature_cols].astype(np.float32)
print(f"Account-level data: {acct.shape[0]:,} customers x {len(feature_cols):,} features")
acct.to_parquet(p("acct_features.parquet"), index=False)

# Slide 4 table: engineered features by category and by type
def agg_type(c):
    for k in ["Response_Rate", "Ever_Response", "Apr_2018", "Ave", "Min", "Max", "Sum", "Chg"]:
        if f"_{k}" in c: return k
    return "Other"
eng = pd.DataFrame({"Feature": feature_cols,
                    "Category": [category_of(c) if c != "N_Months" else "Other" for c in feature_cols],
                    "Type": [agg_type(c) for c in feature_cols]})
by_cat = eng["Category"].value_counts().rename_axis("Category").reset_index(name="# of features")
by_type = eng["Type"].value_counts().rename_axis("Aggregation").reset_index(name="# of features")
by_cat.to_csv(s("slide04_engineered_by_category.csv"), index=False)
by_type.to_csv(s("slide04_engineered_by_type.csv"), index=False)
print(by_cat.to_string(index=False)); print(); print(by_type.to_string(index=False))

## 7. Split: 70% Train, 15% Test 1, 15% Test 2 (stratified by default)

In [ ]:
from sklearn.model_selection import train_test_split
acct = pd.read_parquet(p("acct_features.parquet"))
train, rest = train_test_split(acct, test_size=0.30, stratify=acct["target"], random_state=SEED)
test1, test2 = train_test_split(rest, test_size=0.50, stratify=rest["target"], random_state=SEED)
for name, d in [("train", train), ("test1", test1), ("test2", test2)]:
    d.to_parquet(p(f"{name}.parquet"), index=False)
split = pd.DataFrame({"Sample": ["Train", "Test 1", "Test 2"],
                      "# Obs": [len(train), len(test1), len(test2)],
                      "Default Rate": [train.target.mean(), test1.target.mean(), test2.target.mean()]})
split.to_csv(s("split_summary.csv"), index=False)
print(split.to_string(index=False))

## 8. Feature selection with XGBoost

Missing values are kept as NaN (XGBoost handles them). Two models:
* Model 1: default parameters
* Model 2: 300 trees, learning rate 0.5, max depth 4, 50% rows and 50% features per tree, weight 5 on defaults

Keep every feature with importance > 0.5% in **either** model.

In [ ]:
train, test1, test2 = load_splits()
DROP = ["customer_ID", "target"]
all_feats = [c for c in train.columns if c not in DROP]
X_tr, y_tr = train[all_feats], train["target"]

m1 = XGBClassifier(random_state=SEED, n_jobs=-1, tree_method="hist")
m1.fit(X_tr, y_tr)
m2 = XGBClassifier(n_estimators=300, learning_rate=0.5, max_depth=4, subsample=0.5,
                   colsample_bytree=0.5, scale_pos_weight=5, random_state=SEED, n_jobs=-1, tree_method="hist")
m2.fit(X_tr, y_tr)

fi1 = pd.DataFrame({"Feature": all_feats, "Importance": m1.feature_importances_}).sort_values("Importance", ascending=False)
fi2 = pd.DataFrame({"Feature": all_feats, "Importance": m2.feature_importances_}).sort_values("Importance", ascending=False)
fi1.to_csv(p("feature_importance_model1_default.csv"), index=False)
fi2.to_csv(p("feature_importance_model2_tuned.csv"), index=False)

selected = sorted(set(fi1.loc[fi1.Importance > 0.005, "Feature"]) | set(fi2.loc[fi2.Importance > 0.005, "Feature"]))
json.dump({"selected": selected}, open(p("selected_features.json"), "w"))

with pd.ExcelWriter(s("slide06_feature_importance.xlsx")) as xw:
    fi1.assign(Selected=fi1.Importance > 0.005).to_excel(xw, sheet_name="Model 1 (default)", index=False)
    fi2.assign(Selected=fi2.Importance > 0.005).to_excel(xw, sheet_name="Model 2 (300 trees)", index=False)

for name, m in [("Model 1", m1), ("Model 2", m2)]:
    print(name, "AUC train / test1 / test2:",
          [round(roc_auc_score(d.target, m.predict_proba(d[all_feats])[:, 1]), 4) for d in [train, test1, test2]])
print(f"\nFeatures: {len(all_feats)} engineered -> {(fi1.Importance>0.005).sum()} (model 1) | {(fi2.Importance>0.005).sum()} (model 2) | {len(selected)} kept (union)")

# Slide 6: # features by category, engineered vs selected
cat_all = pd.Series([category_of(c) if c != "N_Months" else "Other" for c in all_feats]).value_counts()
cat_sel = pd.Series([category_of(c) if c != "N_Months" else "Other" for c in selected]).value_counts()
slide6 = pd.DataFrame({"# Engineered": cat_all, "# Selected for Grid Search": cat_sel}).fillna(0).astype(int)
slide6.index.name = "Category"; slide6 = slide6.reset_index()
slide6.to_csv(s("slide06_selected_by_category.csv"), index=False)
print(slide6.to_string(index=False))

# Slide 6 funnel chart
stages = ["Raw features\n(10% sample)", "Engineered\nfeatures", "Importance > 0.5%\nModel 1", "Importance > 0.5%\nModel 2", "Kept\n(union)"]
vals = [len(json.load(open(p("chosen_features.json")))["chosen_features"]), len(all_feats),
        int((fi1.Importance > 0.005).sum()), int((fi2.Importance > 0.005).sum()), len(selected)]
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(stages, vals, color=["#9AA5B1", "#4C6EF5", "#74C0FC", "#74C0FC", "#2B8A3E"])
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v, f"{v:,}", ha="center", va="bottom", fontsize=12)
ax.set_ylabel("# Features"); ax.set_title("Feature Selection Process", fontsize=14)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(s("slide06_feature_selection.png"), dpi=200); plt.show()

## 9. XGBoost grid search (72 models)

The table is saved to Drive after **every** model. If the cell stops, just run it again and it skips the models already done.

In [ ]:
import itertools
train, test1, test2 = load_splits()
selected = json.load(open(p("selected_features.json")))["selected"]

grid = list(itertools.product([50, 100, 300], [0.01, 0.1], [0.5, 0.8], [0.5, 1.0], [1, 5, 10]))
cols = ["# Trees", "LR", "Subsample", "% Features", "Weight of Default", "AUC Train", "AUC Test 1", "AUC Test 2"]
res = pd.read_csv(GRID_XGB) if os.path.exists(GRID_XGB) else pd.DataFrame(columns=cols)
done = set(map(tuple, res[cols[:5]].astype(float).values.tolist()))

t0 = time.time()
for i, (n, lr, sub, colf, w) in enumerate(grid, 1):
    if (float(n), lr, sub, colf, float(w)) in done:
        continue
    m = XGBClassifier(n_estimators=n, learning_rate=lr, subsample=sub, colsample_bytree=colf,
                      scale_pos_weight=w, random_state=SEED, n_jobs=-1, tree_method="hist")
    m.fit(train[selected], train["target"])
    aucs = [roc_auc_score(d["target"], m.predict_proba(d[selected])[:, 1]) for d in [train, test1, test2]]
    res.loc[len(res)] = [n, lr, sub, colf, w] + aucs
    res.to_csv(GRID_XGB, index=False)
    print(f"{i}/{len(grid)}  trees={n} lr={lr} sub={sub} col={colf} w={w}  AUC={[round(a,4) for a in aucs]}  ({time.time()-t0:.0f}s)")
print("Done:", len(res), "models")

## 10. Choose the best XGBoost model (bias vs variance)

* **Bias** = how good the model is → high average AUC.
* **Variance** = how stable it is across samples → low standard deviation of AUC, small gap between Train and Test 2.

Default rule used below: among models whose Train − Test 2 AUC gap is ≤ 0.02, take the one with the highest average AUC. You can override by setting `BEST_XGB_ROW` to a row number from the table.

In [ ]:
rx = summarize_grid(pd.read_csv(GRID_XGB))
BEST_XGB_ROW = pick_best(rx)          # <- override here if you prefer another row, e.g. BEST_XGB_ROW = 17
rx.to_csv(s("slide07_xgb_grid_table.csv"), index_label="Row")
print("Top 10 by average AUC:")
print(rx.sort_values("Avg AUC", ascending=False).head(10).round(4).to_string())
print("\nChosen row:", BEST_XGB_ROW); print(rx.loc[BEST_XGB_ROW].round(4))
grid_scatter(rx, BEST_XGB_ROW, "XGBoost Grid Search (72 models)", "slide08_xgb_scatter.png")

## 10b. Final XGBoost model + rank ordering

In [ ]:
train, test1, test2 = load_splits()
selected = json.load(open(p("selected_features.json")))["selected"]
rx = summarize_grid(pd.read_csv(GRID_XGB))
if "BEST_XGB_ROW" not in globals():
    BEST_XGB_ROW = pick_best(rx)
bp = rx.loc[BEST_XGB_ROW]
xgb_params = dict(n_estimators=int(bp["# Trees"]), learning_rate=float(bp["LR"]), subsample=float(bp["Subsample"]),
                  colsample_bytree=float(bp["% Features"]), scale_pos_weight=float(bp["Weight of Default"]))
xgb_final = XGBClassifier(**xgb_params, random_state=SEED, n_jobs=-1, tree_method="hist")
xgb_final.fit(train[selected], train["target"])
xgb_final.save_model(p("final_xgb_model.json"))
pickle.dump(xgb_final, open(p("final_xgb_model.pkl"), "wb"))
json.dump({"BEST_XGB_ROW": int(BEST_XGB_ROW), **xgb_params}, open(p("final_xgb_params.json"), "w"))

pd_x = {n: xgb_final.predict_proba(d[selected])[:, 1] for n, d in [("train", train), ("test1", test1), ("test2", test2)]}
auc_x = {n: roc_auc_score(d["target"], pd_x[n]) for n, d in [("train", train), ("test1", test1), ("test2", test2)]}
final_x = pd.DataFrame({"Parameter": ["# Trees", "Learning Rate", "Subsample", "% Features", "Weight of Default", "Max Depth (default)",
                                      "AUC Train", "AUC Test 1", "AUC Test 2"],
                        "Value": [xgb_params["n_estimators"], xgb_params["learning_rate"], xgb_params["subsample"], xgb_params["colsample_bytree"],
                                  xgb_params["scale_pos_weight"], 6, round(auc_x["train"], 4), round(auc_x["test1"], 4), round(auc_x["test2"], 4)]})
final_x.to_csv(s("slide09_xgb_final_params.csv"), index=False)
print(final_x.to_string(index=False))
ro_x = rank_order(pd_x["train"], pd_x["test1"], pd_x["test2"], train.target, test1.target, test2.target,
                  "XGBoost Rank Ordering: Default Rate by Score Bin", "slide09_xgb_rank_order.png")
ro_x.to_csv(s("slide09_xgb_rank_order.csv"))
print(ro_x.round(4).to_string())

## 10c. SHAP analysis (Test 2) + top-5 feature summary statistics (slide 4)

In [ ]:
import shap
train, test1, test2 = load_splits()
selected = json.load(open(p("selected_features.json")))["selected"]
xgb_final = pickle.load(open(p("final_xgb_model.pkl"), "rb"))

explainer = shap.TreeExplainer(xgb_final)
sv = explainer(test2[selected])

plt.figure()
shap.plots.beeswarm(sv, max_display=15, show=False)
plt.title("SHAP Beeswarm: Top 15 Drivers of Default (Test 2)", fontsize=13)
plt.tight_layout(); plt.savefig(s("slide10_shap_beeswarm.png"), dpi=200, bbox_inches="tight"); plt.show()

# Waterfall for one customer: by default the Test 2 customer with the highest predicted PD who actually defaulted
pd_t2 = xgb_final.predict_proba(test2[selected])[:, 1]
cand = np.where(test2["target"].values == 1)[0]
OBS = int(cand[np.argmax(pd_t2[cand])])   # <- change to any row number to explain another customer
print("Explaining Test 2 row", OBS, "| predicted PD =", round(float(pd_t2[OBS]), 4), "| actual =", int(test2.target.iloc[OBS]))
plt.figure()
shap.plots.waterfall(sv[OBS], max_display=12, show=False)
plt.title(f"SHAP Waterfall: One Customer (PD = {pd_t2[OBS]:.2%})", fontsize=13)
plt.tight_layout(); plt.savefig(s("slide11_shap_waterfall.png"), dpi=200, bbox_inches="tight"); plt.show()

mean_abs = pd.Series(np.abs(sv.values).mean(axis=0), index=selected).sort_values(ascending=False)
mean_abs.rename("Mean |SHAP|").to_csv(p("shap_importance.csv"))
top5 = mean_abs.head(5).index.tolist()
acct = pd.read_parquet(p("acct_features.parquet"))
rows = []
for f in top5:
    x = acct[f]
    rows.append([f, x.min(), x.quantile(.01), x.quantile(.05), x.median(), x.quantile(.95), x.quantile(.99), x.max(), x.mean(), x.isna().mean()])
top5_tab = pd.DataFrame(rows, columns=["Feature", "Min", "1 Percentile", "5 Percentile", "Median", "95 Percentile", "99 Percentile", "Max", "Mean", "% Missing"])
top5_tab.to_csv(s("slide04_top5_shap_summary.csv"), index=False)
print(top5_tab.round(3).to_string(index=False))

## 11. Neural network data processing

Only the features selected in step 8. Steps (all fitted on **Train**, then applied to Test 1 / Test 2):
1. Missing values → 0
2. Cap and floor at the 1st and 99th percentiles
3. StandardScaler (mean 0, std 1)

In [ ]:
from sklearn.preprocessing import StandardScaler
train, test1, test2 = load_splits()
selected = json.load(open(p("selected_features.json")))["selected"]

def nn_prep_fit(df):
    x = df[selected].fillna(0)
    lo, hi = x.quantile(0.01), x.quantile(0.99)
    x = x.clip(lo, hi, axis=1)
    sc = StandardScaler().fit(x)
    return lo, hi, sc

def nn_prep_apply(df, lo, hi, sc):
    x = df[selected].fillna(0).clip(lo, hi, axis=1)
    return sc.transform(x).astype(np.float32)

lo, hi, sc = nn_prep_fit(train)
Xn = {n: nn_prep_apply(d, lo, hi, sc) for n, d in [("train", train), ("test1", test1), ("test2", test2)]}
yn = {n: d["target"].values for n, d in [("train", train), ("test1", test1), ("test2", test2)]}
pickle.dump({"lo": lo, "hi": hi, "scaler": sc}, open(p("nn_preprocessing.pkl"), "wb"))

proc = pd.DataFrame({"Feature": selected,
                     "% Missing (Train)": train[selected].isna().mean().values,
                     "1st Pct (floor)": lo.values, "99th Pct (cap)": hi.values,
                     "Mean (after cap)": sc.mean_, "Std (after cap)": np.sqrt(sc.var_)})
proc.to_csv(s("slide12_nn_processing.csv"), index=False)
print(proc.head(10).round(3).to_string(index=False))
print("\nAfter scaling, train mean ~", Xn["train"].mean().round(3), "| std ~", Xn["train"].std().round(3))

## 12. Neural network grid search (32 models)

Adam optimizer, binary cross-entropy loss, 20 epochs, defaults for everything else. Dropout "50%" = drop half the nodes; "100%" = keep all nodes (no dropout). Separate loops for 2 and 4 hidden layers. Table saved after every model; re-run to resume.

In [ ]:
import tensorflow as tf
from tensorflow import keras
tf.get_logger().setLevel("ERROR")
cols_nn = ["# HL", "# Node", "Activation", "Dropout", "Batch Size", "AUC Train", "AUC Test 1", "AUC Test 2"]
res_nn = pd.read_csv(GRID_NN) if os.path.exists(GRID_NN) else pd.DataFrame(columns=cols_nn)
done_nn = set((int(a), int(b), c, d, int(e)) for a, b, c, d, e in res_nn[cols_nn[:5]].values.tolist())

def build_nn(n_hidden, nodes, act, dropout_keep):
    keras.utils.set_random_seed(SEED)
    layers = [keras.Input(shape=(Xn["train"].shape[1],))]
    for _ in range(n_hidden):
        layers.append(keras.layers.Dense(nodes, activation=act))
        if dropout_keep == "50%":
            layers.append(keras.layers.Dropout(0.5))
    layers.append(keras.layers.Dense(1, activation="sigmoid"))
    m = keras.Sequential(layers)
    m.compile(optimizer="adam", loss="binary_crossentropy")
    return m

def run_nn(n_hidden, nodes, act, drop, bs):
    global res_nn
    if (n_hidden, nodes, act, drop, bs) in done_nn:
        return
    t0 = time.time()
    m = build_nn(n_hidden, nodes, act, drop)
    m.fit(Xn["train"], yn["train"], epochs=20, batch_size=bs, verbose=0)
    aucs = [roc_auc_score(yn[k], m.predict(Xn[k], batch_size=8192, verbose=0).ravel()) for k in ["train", "test1", "test2"]]
    res_nn.loc[len(res_nn)] = [n_hidden, nodes, act, drop, bs] + aucs
    res_nn.to_csv(GRID_NN, index=False)
    print(f"HL={n_hidden} nodes={nodes} act={act} dropout={drop} batch={bs}  AUC={[round(a,4) for a in aucs]}  ({time.time()-t0:.0f}s)")

NODES, ACTS, DROPS, BATCHES = [4, 6], ["relu", "tanh"], ["50%", "100%"], [100, 10000]

# ---- 2 hidden layers ----
for nodes in NODES:
    for act in ACTS:
        for drop in DROPS:
            for bs in BATCHES:
                run_nn(2, nodes, act, drop, bs)

# ---- 4 hidden layers ----
for nodes in NODES:
    for act in ACTS:
        for drop in DROPS:
            for bs in BATCHES:
                run_nn(4, nodes, act, drop, bs)
print("Done:", len(res_nn), "models")

## 13. Choose the best neural network + final model + rank ordering

In [ ]:
rn = summarize_grid(pd.read_csv(GRID_NN))
BEST_NN_ROW = pick_best(rn)          # <- override here if you prefer another row
rn.to_csv(s("slide13_nn_grid_table.csv"), index_label="Row")
print("Top 10 by average AUC:")
print(rn.sort_values("Avg AUC", ascending=False).head(10).round(4).to_string())
print("\nChosen row:", BEST_NN_ROW); print(rn.loc[BEST_NN_ROW])
grid_scatter(rn, BEST_NN_ROW, "Neural Network Grid Search (32 models)", "slide14_nn_scatter.png")

b = rn.loc[BEST_NN_ROW]
nn_final = build_nn(int(b["# HL"]), int(b["# Node"]), b["Activation"], b["Dropout"])
nn_final.fit(Xn["train"], yn["train"], epochs=20, batch_size=int(b["Batch Size"]), verbose=0)
nn_final.save(p("final_nn_model.keras"))
pd_n = {k: nn_final.predict(Xn[k], batch_size=8192, verbose=0).ravel() for k in ["train", "test1", "test2"]}
auc_n = {k: roc_auc_score(yn[k], pd_n[k]) for k in pd_n}
json.dump({"BEST_NN_ROW": int(BEST_NN_ROW), "hl": int(b["# HL"]), "nodes": int(b["# Node"]), "act": b["Activation"],
           "dropout": b["Dropout"], "batch": int(b["Batch Size"])}, open(p("final_nn_params.json"), "w"))
final_n = pd.DataFrame({"Parameter": ["# Hidden Layers", "# Nodes per Layer", "Activation", "Dropout", "Batch Size",
                                      "Optimizer", "Loss", "Epochs", "AUC Train", "AUC Test 1", "AUC Test 2"],
                        "Value": [int(b["# HL"]), int(b["# Node"]), b["Activation"], b["Dropout"], int(b["Batch Size"]),
                                  "Adam", "Binary cross-entropy", 20, round(auc_n["train"], 4), round(auc_n["test1"], 4), round(auc_n["test2"], 4)]})
final_n.to_csv(s("slide15_nn_final_params.csv"), index=False)
print(final_n.to_string(index=False))
ro_n = rank_order(pd_n["train"], pd_n["test1"], pd_n["test2"], yn["train"], yn["test1"], yn["test2"],
                  "Neural Network Rank Ordering: Default Rate by Score Bin", "slide15_nn_rank_order.png")
ro_n.to_csv(s("slide15_nn_rank_order.csv"))

## 14. XGBoost vs. Neural Network: pick the final model

In [ ]:
train, test1, test2 = load_splits()
selected = json.load(open(p("selected_features.json")))["selected"]
xgb_final = pickle.load(open(p("final_xgb_model.pkl"), "rb"))
pd_x = {n: xgb_final.predict_proba(d[selected])[:, 1] for n, d in [("train", train), ("test1", test1), ("test2", test2)]}
auc_x = {n: roc_auc_score(d["target"], pd_x[n]) for n, d in [("train", train), ("test1", test1), ("test2", test2)]}

from tensorflow import keras
pre = pickle.load(open(p("nn_preprocessing.pkl"), "rb"))
nn_final = keras.models.load_model(p("final_nn_model.keras"))
def nn_X(df):
    x = df[selected].fillna(0).clip(pre["lo"], pre["hi"], axis=1)
    return pre["scaler"].transform(x).astype(np.float32)
pd_n = {n: nn_final.predict(nn_X(d), batch_size=8192, verbose=0).ravel() for n, d in [("train", train), ("test1", test1), ("test2", test2)]}
auc_n = {n: roc_auc_score(d["target"], pd_n[n]) for n, d in [("train", train), ("test1", test1), ("test2", test2)]}

comp = pd.DataFrame({"Model": ["XGBoost", "Neural Network"],
                     "AUC Train": [auc_x["train"], auc_n["train"]],
                     "AUC Test 1": [auc_x["test1"], auc_n["test1"]],
                     "AUC Test 2": [auc_x["test2"], auc_n["test2"]]})
comp["Avg AUC"] = comp[["AUC Train", "AUC Test 1", "AUC Test 2"]].mean(axis=1)
comp["Gap (Train-Test2)"] = comp["AUC Train"] - comp["AUC Test 2"]
comp.to_csv(s("slide16_model_comparison.csv"), index=False)
print(comp.round(4).to_string(index=False))

FINAL_MODEL = "XGBoost" if comp.loc[0, "AUC Test 2"] >= comp.loc[1, "AUC Test 2"] else "Neural Network"   # <- override if you want
print("\nFinal model:", FINAL_MODEL)
pd_final = pd_x if FINAL_MODEL == "XGBoost" else pd_n
json.dump({"FINAL_MODEL": FINAL_MODEL}, open(p("final_model_choice.json"), "w"))

## 15. Strategy: default rate and revenue by threshold

Revenue inputs: one Spend (`S_`) and one Balance (`B_`) feature from our sample, averaged over the last 6 months (Nov 2017 – Apr 2018) = the `_Ave_6` features we already built.
Default choice: the S_ and B_ feature with the fewest missing values (change `SPEND_RAW` / `BAL_RAW` if you want).
Missing or negative averages are set to 0 (a customer cannot have negative spend).

Monthly revenue = B_Ave × 0.02 + S_Ave × 0.001; 12-month revenue = 12 × monthly; revenue = 0 for customers who default.

In [ ]:
acct = pd.read_parquet(p("acct_features.parquet"))
g = json.load(open(p("column_groups.json")))
s_opts = [c for c in g["nums"] if c.startswith("S_")]
b_opts = [c for c in g["nums"] if c.startswith("B_")]
miss = lambda c: acct[f"{c}_Ave_6"].isna().mean()
SPEND_RAW = min(s_opts, key=miss)     # <- or set e.g. "S_3"
BAL_RAW   = min(b_opts, key=miss)     # <- or set e.g. "B_1"
print("Spend options:", s_opts, "| Balance options:", b_opts)
print("Using spend =", SPEND_RAW, "| balance =", BAL_RAW)

def strategy_data(df, pds):
    return pd.DataFrame({"target": df["target"].values, "PD": pds,
                         "Balance": df[f"{BAL_RAW}_Ave_6"].fillna(0).clip(lower=0).values,
                         "Spend": df[f"{SPEND_RAW}_Ave_6"].fillna(0).clip(lower=0).values})

def strategy(data, target, pd_col, balance_col, spend_col, threshold):
    """Accept applicants with PD < threshold.
    Returns (portfolio default rate, portfolio expected 12-month revenue)."""
    accepted = data[data[pd_col] < threshold]
    if len(accepted) == 0:
        return np.nan, 0.0
    default_rate = accepted[target].mean()
    monthly_rev = accepted[balance_col] * 0.02 + accepted[spend_col] * 0.001
    revenue = (12 * monthly_rev * (1 - accepted[target])).sum()     # 0 revenue for defaulters
    return default_rate, revenue

S = {"Train": strategy_data(train, pd_final["train"]),
     "Test 1": strategy_data(test1, pd_final["test1"]),
     "Test 2": strategy_data(test2, pd_final["test2"])}

def strategy_table(thresholds):
    rows = []
    for t in thresholds:
        r = [t]
        for k, d in S.items():
            dr, rev = strategy(d, "target", "PD", "Balance", "Spend", t)
            r += [int((d["PD"] < t).sum()), dr, rev]
        rows.append(r)
    cols = ["Threshold"] + [f"{k} {m}" for k in S for m in ["#Total", "Default Rate", "Revenue"]]
    return pd.DataFrame(rows, columns=cols)

tab10 = strategy_table([round(x, 1) for x in np.arange(0.1, 1.01, 0.1)])
tab10.to_csv(s("slide17_threshold_table.csv"), index=False)
print(tab10.round(4).to_string(index=False))

# fine grid on TRAIN only, to choose the two strategies
fine = strategy_table(np.round(np.arange(0.01, 1.0001, 0.01), 2))
fine.to_csv(p("strategy_fine_grid.csv"), index=False)

### Choose the two thresholds (decision)

Rule used by default (professor's instruction: the 10% limit must hold on **Train, Test 1 and Test 2**):
* **Aggressive** = highest threshold where the default rate is ≤ 10% in **all three** samples. Max revenue within the limit.
* **Conservative** = highest threshold where the default rate is ≤ `CONSERVATIVE_TARGET_DR` (default 5%) in **all three** samples. Fewer approvals, safer portfolio.

Change `CONSERVATIVE_TARGET_DR` or set `AGGRESSIVE_T` / `CONSERVATIVE_T` by hand if you want different strategies.

In [ ]:
LIMIT_DR = 0.10
CONSERVATIVE_TARGET_DR = 0.05

# worst (highest) default rate across Train, Test 1 and Test 2 for each threshold
fine["Max Default Rate (all samples)"] = fine[["Train Default Rate", "Test 1 Default Rate", "Test 2 Default Rate"]].max(axis=1)
ok = fine[fine["Max Default Rate (all samples)"] <= LIMIT_DR]
AGGRESSIVE_T = ok["Threshold"].max()
okc = fine[fine["Max Default Rate (all samples)"] <= CONSERVATIVE_TARGET_DR]
CONSERVATIVE_T = okc["Threshold"].max()
print(fine.loc[fine["Threshold"].isin([CONSERVATIVE_T, AGGRESSIVE_T]),
               ["Threshold", "Train Default Rate", "Test 1 Default Rate", "Test 2 Default Rate"]].round(4).to_string(index=False))
print("Aggressive threshold:", AGGRESSIVE_T, "| Conservative threshold:", CONSERVATIVE_T)

final_strat = strategy_table([CONSERVATIVE_T, AGGRESSIVE_T])
final_strat.insert(0, "Strategy", [f"Conservative (PD < {CONSERVATIVE_T})", f"Aggressive (PD < {AGGRESSIVE_T})"])
n_all = {k: len(d) for k, d in S.items()}
for k in S:
    final_strat[f"{k} Approval Rate"] = final_strat[f"{k} #Total"] / n_all[k]
final_strat.to_csv(s("slide01_17_strategy_table.csv"), index=False)
print(final_strat.round(4).T.to_string())

# chart: default rate and revenue vs threshold (Train)
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(fine["Threshold"], fine["Train Default Rate"], c="#E03131", lw=2, label="Default rate")
ax1.axhline(LIMIT_DR, ls="--", c="#E03131", lw=1); ax1.text(0.01, LIMIT_DR + 0.003, "10% limit", color="#E03131")
ax1.set_xlabel("PD threshold (accept if PD < threshold)"); ax1.set_ylabel("Portfolio default rate", color="#E03131")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax2 = ax1.twinx()
ax2.plot(fine["Threshold"], fine["Train Revenue"], c="#2B8A3E", lw=2, label="Revenue")
ax2.set_ylabel("Expected 12-month revenue", color="#2B8A3E")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
for t, lbl, ha, yf in [(CONSERVATIVE_T, "Conservative", "right", 0.95), (AGGRESSIVE_T, "Aggressive", "left", 0.80)]:
    ax1.axvline(t, c="gray", ls=":"); ax1.text(t, ax1.get_ylim()[1]*yf, f" {lbl} ({t}) ", fontsize=10, ha=ha)
ax1.set_title("Train: Default Rate and Revenue by Threshold", fontsize=14)
ax1.spines[["top"]].set_visible(False); ax2.spines[["top"]].set_visible(False)
plt.tight_layout(); plt.savefig(s("slide17_threshold_curve.png"), dpi=200); plt.show()

## Done: list of slide files

In [ ]:
for f in sorted(os.listdir(SLIDES)):
    print(f)